# Notebook 7 — Encoding Categorical Variables
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

df = pd.read_csv("telco_churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
print(f"Dataset loaded: {df.shape[0]:,} rows")


Dataset loaded: 7,043 rows


---
## 1. What is Categorical Encoding?

### Understand
Categorical encoding converts text/category values into numbers, since virtually every
ML algorithm requires numeric input. *How* to encode depends entirely on what kind of
category it is — treating them all the same way is the single most common encoding
mistake.

### Demonstrate
**AI/ML use case:** Feeding `Contract` (text) directly into `LogisticRegression` raises
an error immediately — encoding isn't optional polish, it's a hard requirement.


---
## 2. Nominal, 3. Ordinal, and 4. Binary Variables

### Understand
- **Nominal**: categories with NO natural order (`InternetService`: DSL/Fiber/No).
- **Ordinal**: categories WITH a natural order (`Contract`: Month-to-month < One year <
  Two year).
- **Binary**: exactly two categories (`gender`, `Partner`, `Churn`).

Each type has a different *correct* encoding — using an ordinal encoding on a nominal
variable would falsely imply an order that doesn't exist.

### Implement


In [2]:
variable_types = {
    'Nominal': ['gender', 'InternetService', 'PaymentMethod', 'MultipleLines'],
    'Ordinal': ['Contract'],
    'Binary': ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn'],
}
for vtype, cols in variable_types.items():
    print(f"{vtype}: {cols}")


Nominal: ['gender', 'InternetService', 'PaymentMethod', 'MultipleLines']
Ordinal: ['Contract']
Binary: ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']


---
## 5. Label Encoding

### Understand
Assigns each category an arbitrary integer (0, 1, 2, ...). **Risk:** for a NOMINAL
variable, this falsely implies a numeric order/distance between categories that doesn't
exist — a model might learn "Fiber optic > DSL" purely from the encoding, not the data.

### When appropriate: binary variables (only 2 categories, so there's no false "order"
implied), or tree-based models that don't assume numeric distance matters.

### Implement


In [3]:
le = LabelEncoder()
df['Churn_encoded'] = le.fit_transform(df['Churn'])
print(f"Churn label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Demonstrating the RISK on a nominal variable with 3+ categories
le_internet = LabelEncoder()
internet_encoded = le_internet.fit_transform(df['InternetService'])
print(f"\nInternetService label encoding (risky if used this way): {dict(zip(le_internet.classes_, le_internet.transform(le_internet.classes_)))}")
print("A linear model would now treat 'No'=2 as 'twice' Fiber optic=1 — meaningless.")


Churn label encoding: {'No': np.int64(0), 'Yes': np.int64(1)}

InternetService label encoding (risky if used this way): {'DSL': np.int64(0), 'Fiber optic': np.int64(1), 'No': np.int64(2)}
A linear model would now treat 'No'=2 as 'twice' Fiber optic=1 — meaningless.


**Decision:** Label encoding is appropriate for the binary `Churn` target (no false
order risk), but NOT used for `InternetService` in this notebook's final encoding — that
risk is exactly why One-Hot Encoding (Topic 7) is used for nominal variables instead.


---
## 6. Ordinal Encoding

### Understand
Like label encoding, but the integer order is *deliberately* chosen to match a real
business ordering — appropriate specifically for genuinely ordinal variables.

### Implement


In [4]:
contract_order = [['Month-to-month', 'One year', 'Two year']]
ordinal_encoder = OrdinalEncoder(categories=contract_order)
df['Contract_encoded'] = ordinal_encoder.fit_transform(df[['Contract']])

print(df[['Contract', 'Contract_encoded']].drop_duplicates().sort_values('Contract_encoded'))


          Contract  Contract_encoded
0   Month-to-month               0.0
1         One year               1.0
11        Two year               2.0


**Decision: SELECTED for `Contract`.** Unlike label encoding InternetService, this
0/1/2 ordering is meaningful — a model CAN legitimately use "more than" comparisons here,
since Two year genuinely represents more commitment than One year.


---
## 7. One-Hot Encoding & 8. Dummy Variables

### Understand
Creates one new binary (0/1) column per category, with no implied order — the correct
default for nominal variables. "Dummy variables" specifically refers to using
`drop_first=True`, dropping one category to avoid perfect multicollinearity between the
new columns (since if you know all-but-one, the last is fully determined).

### Implement


In [5]:
nominal_cols = ['gender', 'InternetService', 'PaymentMethod', 'MultipleLines']
one_hot = pd.get_dummies(df[nominal_cols], drop_first=True)
print(f"Original columns: {len(nominal_cols)} -> One-hot columns: {one_hot.shape[1]}")
print(one_hot.columns.tolist())


Original columns: 4 -> One-hot columns: 8
['gender_Male', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'MultipleLines_No phone service', 'MultipleLines_Yes']


**Decision: SELECTED for all nominal columns.** `drop_first=True` avoids the
"dummy variable trap" (perfect multicollinearity) while still fully representing every
category through the remaining columns.


---
## 9. Frequency Encoding

### Understand
Replaces each category with how often it appears in the data — useful for high-cardinality
columns where one-hot would create too many new columns, though not needed for this
dataset's uniformly low-cardinality columns (Sprint 4, Notebook 9).

### Implement (illustrative on PaymentMethod)


In [6]:
freq_map = df['PaymentMethod'].value_counts(normalize=True)
df['PaymentMethod_freq'] = df['PaymentMethod'].map(freq_map)
print(df[['PaymentMethod', 'PaymentMethod_freq']].drop_duplicates().sort_values('PaymentMethod_freq', ascending=False))


               PaymentMethod  PaymentMethod_freq
0           Electronic check            0.335794
1               Mailed check            0.228880
3  Bank transfer (automatic)            0.219225
6    Credit card (automatic)            0.216101


**Decision: Demonstrated, not required for this dataset** — every categorical
column here has only 2-4 categories (confirmed low cardinality, Sprint 4), so one-hot
encoding never becomes unwieldy; frequency encoding's main advantage doesn't apply.


---
## 10. Target Encoding

### Understand
Replaces each category with the mean target value for that category (e.g., each
`Contract` type's churn rate). **Major risk: target leakage** if computed on the full
dataset before a train/test split — the encoding would leak target information into the
"features," inflating validation performance unrealistically (this is examined in full
in Notebook 13).

### Implement (illustrative, with the leakage risk explicitly flagged)


In [7]:
df['Churn_numeric'] = (df['Churn'] == 'Yes').astype(int)
target_encoded_risky = df.groupby('Contract')['Churn_numeric'].transform('mean')
print("Target-encoded Contract (DANGEROUS if computed on the full/train+test dataset):")
print(df[['Contract']].assign(target_enc=target_encoded_risky).drop_duplicates())
print("\nCorrect practice: fit this encoding ONLY on the training fold, then apply to validation/test — covered fully in Notebook 13.")


Target-encoded Contract (DANGEROUS if computed on the full/train+test dataset):
          Contract  target_enc
0   Month-to-month    0.427097
1         One year    0.112695
11        Two year    0.028319

Correct practice: fit this encoding ONLY on the training fold, then apply to validation/test — covered fully in Notebook 13.


**Decision: Demonstrated for completeness, NOT used in this notebook's final
encoded dataset** — target encoding is powerful but requires careful, split-aware
implementation that belongs with Notebook 12-13's train/test discipline, not here.


---
## 11. Handling Unknown Categories

### Understand
A category seen at prediction time but never seen during training (e.g., a brand-new
`PaymentMethod` introduced after the model was trained) will break a naive encoder.
`OneHotEncoder(handle_unknown='ignore')` is the standard defense.

### Implement


In [8]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(df[['PaymentMethod']])

new_data = pd.DataFrame({'PaymentMethod': ['Cryptocurrency']})   # a category never seen in training
result = ohe.transform(new_data)
print(f"Unknown category 'Cryptocurrency' handled without crashing: {result}")
print("(All-zero row — the encoder correctly recognizes it as 'none of the known categories', instead of raising an error.)")


Unknown category 'Cryptocurrency' handled without crashing: [[0. 0. 0. 0.]]
(All-zero row — the encoder correctly recognizes it as 'none of the known categories', instead of raising an error.)


**Finding:** `handle_unknown='ignore'` prevents a pipeline from crashing on future,
unseen categories — an important production-readiness detail that a raw `pd.get_dummies()`
call (Topic 7) does NOT provide, which is why scikit-learn's `OneHotEncoder` is the safer
choice inside a production Pipeline (Notebook 14).


---
## 12. High-Cardinality Categories

### Understand
A column with many distinct values (hundreds or thousands) makes one-hot encoding
impractical (too many new columns) — this dataset has none (Sprint 4, Notebook 9
confirmed every genuine categorical column has 2-4 categories), so this is documented as
a non-issue here, with the general mitigation strategies noted for future reference.

### Implement


In [9]:
cardinality_check = df.select_dtypes(include='object').drop(columns=['customerID', 'Churn']).nunique()
print(cardinality_check.sort_values(ascending=False))
print(f"\nMax cardinality among true categorical columns: {cardinality_check.max()} — no high-cardinality concern.")
print("(If it existed: frequency encoding, target encoding, or hashing would be the mitigation.)")


PaymentMethod       4
InternetService     3
OnlineSecurity      3
DeviceProtection    3
TechSupport         3
OnlineBackup        3
MultipleLines       3
StreamingTV         3
StreamingMovies     3
Contract            3
Dependents          2
PhoneService        2
gender              2
Partner             2
PaperlessBilling    2
dtype: int64

Max cardinality among true categorical columns: 4 — no high-cardinality concern.
(If it existed: frequency encoding, target encoding, or hashing would be the mitigation.)


/tmp/ipykernel_658/2655337364.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cardinality_check = df.select_dtypes(include='object').drop(columns=['customerID', 'Churn']).nunique()


---
## Final Encoded Feature Set

### Documentation (Problem / Analysis / Technique / Reason / Implementation / Result / Impact)
- **Problem:** 16 categorical columns need numeric representation before modeling.
- **Analysis:** Classified each by type (Topic 2-4): 1 ordinal, 14 nominal, `Churn` is the
  binary target.
- **Technique Selected:** Ordinal encoding for `Contract`; one-hot (`drop_first=True`)
  for all nominal columns; label encoding for `Churn`.
- **Reason:** Matches each column's true structure — avoids both false-order risk
  (label encoding a nominal column) and unnecessary column explosion (all columns are
  low-cardinality, so one-hot stays manageable).
- **Implementation:** below, combining all pieces into one final encoded DataFrame.
- **Result:** see the shape comparison below.
- **Impact:** Every remaining column is now purely numeric, ready for scaling (Notebook 8)
  and modeling.


In [10]:
nominal_all = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
               'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
               'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'PaymentMethod']

encoded_nominal = pd.get_dummies(df[nominal_all], drop_first=True)
final_encoded = pd.concat([
    df[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Contract_encoded']],
    encoded_nominal,
    df[['Churn_encoded']]
], axis=1)

print(f"Before encoding: {df.shape[1]} columns")
print(f"After encoding : {final_encoded.shape[1]} columns, all numeric")
print(f"All numeric now: {final_encoded.dtypes.apply(lambda d: d.kind in 'ifb').all()}")


Before encoding: 25 columns
After encoding : 30 columns, all numeric
All numeric now: True


---
## Summary

| Column Type | Technique | Example |
|---|---|---|
| Binary target | Label Encoding | `Churn` -> 0/1 |
| Ordinal | Ordinal Encoding | `Contract` -> 0/1/2 (meaningful order) |
| Nominal (all others) | One-Hot Encoding (`drop_first=True`) | `InternetService`, `PaymentMethod`, etc. |
| High-cardinality | N/A for this dataset | None found (max cardinality = 4) |

**Risk explicitly avoided:** label-encoding a nominal variable like `InternetService`
would have implied a false numeric order — demonstrated directly, not just asserted.

**Next notebook:** `08_Feature_Scaling.ipynb` — scaling the now-fully-numeric feature set.
